**Library**

In [7]:
import sys
import os
import pandas as pd
import itertools

import mlflow
import mlflow.sklearn
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import warnings

**Load Data**

In [2]:
# Cell 1: Persiapan Environment & Pengambilan Data (Anti-Cache Jupyter)
import sys
import os
import importlib

# 1. INJEKSI KREDENSIAL LANGSUNG KE MEMORI (Bypass file .env)
os.environ["DB_USER"] = "postgres"
os.environ["DB_PASS"] = "laygenda102"  # PENTING: Ganti dengan password asli Anda!
os.environ["DB_HOST"] = "localhost"
os.environ["DB_PORT"] = "5432"
os.environ["DB_NAME"] = "SeismoCluster"

# 2. SETUP PATH UNTUK IMPORT BACKEND
backend_path = os.path.abspath('../backend')
if backend_path not in sys.path:
    sys.path.append(backend_path)

# 3. FORCE RELOAD MODUL 
import app.config.database
importlib.reload(app.config.database) # PAKSA Python mereset variabel dan membaca ulang os.environ
from app.config.database import SessionLocal

import app.models.pipeline
importlib.reload(app.models.pipeline)
from app.models.pipeline import SeismoDataPipeline

# 4. EKSEKUSI PIPELINE
db = SessionLocal()
pipeline = SeismoDataPipeline()

try:
    print("Membuka koneksi PostgreSQL dan menyedot data...")
    df_raw = pipeline.load_from_db(db)
    
    print("Melakukan Transformasi dan Scaling (4 Dimensi)...")
    df_processed = pipeline.prepare_features(df_raw)
    
    # Ekstraksi fitur yang akan dimasukkan ke dalam algoritma ML
    X_scaled = df_processed[['latitude_scaled', 'longitude_scaled', 'depth_scaled', 'magnitude_scaled']].values
    print(f"Data siap dikompetisikan! Jumlah observasi: {len(X_scaled)} baris.")
    
except Exception as e:
    print(f"Terjadi kesalahan Database: {e}")
finally:
    db.close() # Menutup sesi agar database tidak memakan banyak RAM

Membuka koneksi PostgreSQL dan menyedot data...
Melakukan Transformasi dan Scaling (4 Dimensi)...
Data siap dikompetisikan! Jumlah observasi: 12490 baris.


**Uji coba K-Means**

In [ ]:
warnings.filterwarnings("ignore")

# 1. SETUP MLFLOW TRACKING
# Menyambungkan notebook ke server MLflow lokal
mlflow.set_tracking_uri("http://localhost:5000")

# Membuat/Membuka "map" khusus untuk eksperimen K-Means
mlflow.set_experiment("SeismoCluster_KMeans_Tuning")

print("Memulai Hyperparameter Tuning untuk K-Means...")
print("-" * 50)

# 2. DEFINISI RENTANG HYPERPARAMETER
# Kita akan menguji pembagian zona gempa dari 3 hingga 8 cluster
K_range = [3, 4, 5, 6, 7, 8]

# 3. LOOPING EKSPERIMEN OTOMATIS (Sesuai instruksi tugas dosen)
for k in K_range:
    # Membuka satu sesi pencatatan di MLflow untuk setiap nilai K
    with mlflow.start_run(run_name=f"KMeans_K{k}"):
        
        # --- A. TRAINING MODEL ---
        # Menambahkan n_init=10 agar hasil konsisten dan solid
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X_scaled)
        
        # --- B. EVALUASI METRIK ---
        sil_score = silhouette_score(X_scaled, labels)
        db_score = davies_bouldin_score(X_scaled, labels)
        ch_score = calinski_harabasz_score(X_scaled, labels)
        
        # --- C. LOGGING PARAMETER & METRIK KE MLFLOW ---
        # Mencatat hyperparameter yang sedang diuji
        mlflow.log_param("algorithm", "K-Means")
        mlflow.log_param("n_clusters", k)
        
        # Mencatat hasil ujian (metric)
        mlflow.log_metric("silhouette_score", sil_score)
        mlflow.log_metric("davies_bouldin_index", db_score)
        mlflow.log_metric("calinski_harabasz", ch_score)
        
        # --- D. SAVE ARTIFACT (Simpan Model Fisik) ---
        mlflow.sklearn.log_model(kmeans, "model")
        
        print(f"Selesai K={k} | Silhouette: {sil_score:.4f} | DB Index: {db_score:.4f} | CH Score: {ch_score:.1f}")

print("-" * 50)
print("Tuning K-Means Selesai! Seluruh bukti riset telah tersimpan di MLflow.")

Memulai Hyperparameter Tuning untuk K-Means...
--------------------------------------------------


  File "c:\Users\laygenda surya putra\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "c:\Users\laygenda surya putra\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "c:\Users\laygenda surya putra\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\laygenda surya putra\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1456, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
2026/05/07 09:06:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 09:06:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these 

Selesai K=3 | Silhouette: 0.3154 | DB Index: 1.2977 | CH Score: 4111.3
🏃 View run KMeans_K3 at: http://localhost:5000/#/experiments/1/runs/56d7372a6f65427a88e9e3e12c627eef
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/05/07 09:06:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 09:06:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Selesai K=4 | Silhouette: 0.3316 | DB Index: 1.0791 | CH Score: 4730.7
🏃 View run KMeans_K4 at: http://localhost:5000/#/experiments/1/runs/3f5c11d0d01745e3a65013adc9599145
🧪 View experiment at: http://localhost:5000/#/experiments/1


...

Selesai K=6 | Silhouette: 0.3317 | DB Index: 1.0128 | CH Score: 4996.8
🏃 View run KMeans_K6 at: http://localhost:5000/#/experiments/1/runs/49dc0b75693448909eeadea6d8f4a28a
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/05/07 09:07:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 09:07:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Selesai K=7 | Silhouette: 0.3214 | DB Index: 1.0057 | CH Score: 4923.2
🏃 View run KMeans_K7 at: http://localhost:5000/#/experiments/1/runs/2bce1830b2414a369b7227129969e1e3
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/05/07 09:07:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 09:07:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Selesai K=8 | Silhouette: 0.3216 | DB Index: 1.0255 | CH Score: 4852.3
🏃 View run KMeans_K8 at: http://localhost:5000/#/experiments/1/runs/91f1e3ad44a64c81a2af8708d8393deb
🧪 View experiment at: http://localhost:5000/#/experiments/1
--------------------------------------------------
Tuning K-Means Selesai! Seluruh bukti riset telah tersimpan di MLflow.


**Uji coba DBSCAN**

Melakukan otomatisasi pencarian paramater terbaik untuk eps (epsilon) dan min_samples agar model dapat membedakan antara klaster gempa yang padat dan kejadian gempa yang bersifat anomali atau terisolasi.

In [ ]:
# 1. SETUP EXPERIMENT
mlflow.set_experiment("SeismoCluster_Clustering_Competition")

print("Memulai Hyperparameter Tuning untuk DBSCAN...")
print("-" * 50)

# 2. DEFINISI RUANG PARAMETER (Hyperparameter Grid)
# eps: Jarak maksimal antar dua titik untuk dianggap satu lingkungan
# min_samples: Jumlah minimum titik untuk membentuk sebuah cluster padat
param_grid = {
    "eps": [0.1, 0.15, 0.2, 0.25, 0.3],
    "min_samples": [10, 15, 20]
}

# Membuat semua kombinasi pasangan eps dan min_samples
combinations = list(itertools.product(param_grid["eps"], param_grid["min_samples"]))

# 3. LOOPING EKSPERIMEN OTOMATIS
for eps, min_samples in combinations:
    # Membuka sesi run dengan nama yang deskriptif
    with mlflow.start_run(run_name=f"DBSCAN_e{eps}_s{min_samples}"):
        
        # --- A. TRAINING MODEL ---
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X_scaled)
        
        # Hitung jumlah cluster yang terbentuk (mengabaikan noise -1)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        # --- B. EVALUASI METRIK ---
        # Catatan: Silhouette score hanya bisa dihitung jika ada minimal 2 cluster
        if n_clusters > 1:
            sil_score = silhouette_score(X_scaled, labels)
            db_score = davies_bouldin_score(X_scaled, labels)
            ch_score = calinski_harabasz_score(X_scaled, labels)
        else:
            sil_score, db_score, ch_score = 0, 0, 0
        
        # --- C. LOGGING KE MLFLOW ---
        mlflow.log_param("algorithm", "DBSCAN")
        mlflow.log_param("eps", eps)
        mlflow.log_param("min_samples", min_samples)
        
        mlflow.log_metric("n_clusters", n_clusters)
        mlflow.log_metric("n_noise", n_noise)
        mlflow.log_metric("silhouette_score", sil_score)
        mlflow.log_metric("davies_bouldin_index", db_score)
        mlflow.log_metric("calinski_harabasz", ch_score)
        
        # --- D. SAVE ARTIFACT (Log Model) ---
        mlflow.sklearn.log_model(dbscan, "model")
        
        print(f"eps={eps} | samples={min_samples} | Clusters: {n_clusters} | Noise: {n_noise} | Silh: {sil_score:.4f}")

print("-" * 50)
print("Tuning DBSCAN Selesai! Silakan bandingkan hasilnya dengan K-Means di MLflow UI.")

2026/05/07 09:19:21 INFO mlflow.tracking.fluent: Experiment with name 'SeismoCluster_Clustering_Competition' does not exist. Creating a new experiment.


Memulai Hyperparameter Tuning untuk DBSCAN...
--------------------------------------------------


2026/05/07 09:19:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 09:19:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/07 09:19:30 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


eps=0.1 | samples=10 | Clusters: 94 | Noise: 9076 | Silh: -0.4332
🏃 View run DBSCAN_e0.1_s10 at: http://localhost:5000/#/experiments/2/runs/c46a5e3a69b542b6ac79cbaf9a82f7f3
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/05/07 09:19:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 09:19:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/07 09:19:42 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


...

eps=0.3 | samples=10 | Clusters: 12 | Noise: 1348 | Silh: -0.3848
🏃 View run DBSCAN_e0.3_s10 at: http://localhost:5000/#/experiments/2/runs/99ce7f39e9e441e9a54798ed36bb34c4
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/05/07 09:21:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 09:21:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/07 09:21:03 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


eps=0.3 | samples=15 | Clusters: 9 | Noise: 2032 | Silh: -0.2818
🏃 View run DBSCAN_e0.3_s15 at: http://localhost:5000/#/experiments/2/runs/3d64e7ee96ba4daeb59d7361d14cb09c
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/05/07 09:21:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 09:21:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/07 09:21:10 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


eps=0.3 | samples=20 | Clusters: 5 | Noise: 2661 | Silh: -0.1102
🏃 View run DBSCAN_e0.3_s20 at: http://localhost:5000/#/experiments/2/runs/9d8431a18ef449c19bac9f402e9485d3
🧪 View experiment at: http://localhost:5000/#/experiments/2
--------------------------------------------------
Tuning DBSCAN Selesai! Silakan bandingkan hasilnya dengan K-Means di MLflow UI.


**Uji coba Hierarchical**

Melakukan otomatisasi pencarian parameter optimal untuk n_clusters (jumlah zona) dan linkage (kriteria penggabungan) guna melihat struktur hubungan bertingkat antar wilayah gempa.

In [8]:
# 1. SETUP EXPERIMENT
# Kita tetap menggunakan arena kompetisi yang sama agar mudah dibandingkan di UI
mlflow.set_experiment("SeismoCluster_Clustering_Competition")

print("Memulai Hyperparameter Tuning untuk Hierarchical Clustering...")
print("-" * 50)

# 2. DEFINISI RUANG PARAMETER (Hyperparameter Grid)
# n_clusters: Jumlah zona yang ingin dibentuk
# linkage: Metode untuk menghitung jarak antar kelompok saat digabungkan
param_grid_hc = {
    "n_clusters": [3, 4, 5, 6],
    "linkage": ["ward", "complete", "average"]
}

# Membuat semua kombinasi pasangan n_clusters dan linkage
combinations_hc = list(itertools.product(param_grid_hc["n_clusters"], param_grid_hc["linkage"]))

# 3. LOOPING EKSPERIMEN OTOMATIS (Sesuai instruksi tugas dosen)
for n_clusters, linkage in combinations_hc:
    # Membuka sesi run dengan nama yang deskriptif untuk setiap kombinasi
    with mlflow.start_run(run_name=f"HC_n{n_clusters}_l_{linkage}"):
        
        # --- A. TRAINING MODEL ---
        # AgglomerativeClustering adalah metode Hierarchical paling populer
        hc = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage)
        labels = hc.fit_predict(X_scaled)
        
        # --- B. EVALUASI METRIK (Sesuai tabel rencana Anda) ---
        sil_score = silhouette_score(X_scaled, labels)
        db_score = davies_bouldin_score(X_scaled, labels)
        ch_score = calinski_harabasz_score(X_scaled, labels)
        
        # --- C. LOGGING KE MLFLOW ---
        mlflow.log_param("algorithm", "Hierarchical")
        mlflow.log_param("n_clusters", n_clusters)
        mlflow.log_param("linkage", linkage)
        
        mlflow.log_metric("silhouette_score", sil_score)
        mlflow.log_metric("davies_bouldin_index", db_score)
        mlflow.log_metric("calinski_harabasz", ch_score)
        
        # --- D. SAVE ARTIFACT (Log Model) ---
        mlflow.sklearn.log_model(hc, "model")
        
        print(f"n={n_clusters} | Linkage: {linkage:8} | Silh: {sil_score:.4f} | DB Index: {db_score:.4f}")

print("-" * 50)
print("Tuning Hierarchical Clustering Selesai! Semua data riset tersimpan di MLflow.")

Memulai Hyperparameter Tuning untuk Hierarchical Clustering...
--------------------------------------------------


2026/05/07 09:31:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 09:31:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/07 09:31:42 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


n=3 | Linkage: ward     | Silh: 0.3237 | DB Index: 1.2241
🏃 View run HC_n3_l_ward at: http://localhost:5000/#/experiments/2/runs/1feb3d6567504b0481585ae19af286b3
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/05/07 09:31:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 09:31:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/07 09:31:57 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


n=3 | Linkage: complete | Silh: 0.2599 | DB Index: 1.2117
🏃 View run HC_n3_l_complete at: http://localhost:5000/#/experiments/2/runs/e77384bbd67047fba08e042245b8a147
🧪 View experiment at: http://localhost:5000/#/experiments/2


...

n=6 | Linkage: ward     | Silh: 0.2980 | DB Index: 1.0399
🏃 View run HC_n6_l_ward at: http://localhost:5000/#/experiments/2/runs/0593d5fa3a5a40c997036279d0e602fb
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/05/07 09:34:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 09:34:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/07 09:34:13 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


n=6 | Linkage: complete | Silh: 0.1791 | DB Index: 1.1198
🏃 View run HC_n6_l_complete at: http://localhost:5000/#/experiments/2/runs/eec9bda9e1bb4edeb6b75bcf29075d8b
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/05/07 09:34:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 09:34:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/07 09:34:27 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


n=6 | Linkage: average  | Silh: 0.2748 | DB Index: 0.8752
🏃 View run HC_n6_l_average at: http://localhost:5000/#/experiments/2/runs/2931e69285074f61a764d8a360a9c681
🧪 View experiment at: http://localhost:5000/#/experiments/2
--------------------------------------------------
Tuning Hierarchical Clustering Selesai! Semua data riset tersimpan di MLflow.
